# 02 — Demand Forecasting

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

demand = pd.read_csv("../data/raw/weekly_demand.csv", parse_dates=["week_start"])
demand = demand.sort_values(["sku_id","week_start"])

In [ ]:
def mape_nonzero(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100

rows = []
forecast_rows = []

for sku_id, g in demand.groupby("sku_id"):
    g = g.sort_values("week_start").reset_index(drop=True)
    train, test = g.iloc[:-13].copy(), g.iloc[-13:].copy()
    y_train = train["demand_units"].astype(float)
    y_test = test["demand_units"].astype(float)

    naive = np.repeat(y_train.iloc[-1], len(test))
    ma4 = np.repeat(y_train.tail(4).mean(), len(test))

    try:
        model = ExponentialSmoothing(
            y_train, trend="add", seasonal="add",
            seasonal_periods=52, initialization_method="estimated"
        ).fit(optimized=True)
        hw = np.clip(model.forecast(len(test)).values, 0, None)
    except Exception:
        hw = ma4.copy()

    methods = {"Naive":naive, "Moving Average (4W)":ma4, "Holt-Winters":hw}
    for method, pred in methods.items():
        rows.append([
            sku_id, method,
            mean_absolute_error(y_test,pred),
            sqrt(mean_squared_error(y_test,pred)),
            mape_nonzero(y_test,pred)
        ])
        for dt,a,f in zip(test["week_start"],y_test,pred):
            forecast_rows.append([sku_id,dt,method,a,float(f)])

metrics = pd.DataFrame(rows, columns=["sku_id","method","mae","rmse","mape_pct"])
forecasts = pd.DataFrame(forecast_rows, columns=["sku_id","week_start","method","actual_demand","forecast_demand"])
metrics.to_csv("../data/processed/forecast_metrics.csv", index=False)
forecasts.to_csv("../data/processed/forecast_holdout.csv", index=False)

best = metrics.loc[metrics.groupby("sku_id")["rmse"].idxmin()].sort_values("sku_id")
best.to_csv("../data/processed/best_forecast_method_by_sku.csv", index=False)

display(metrics.groupby("method")[["mae","rmse","mape_pct"]].mean().sort_values("rmse"))
display(best)